In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
HOME_DIR = Path(f"{os.getenv('HOME', '')}")

In [34]:
def get_time_avg_metrics(filename: Path, metric_name: str,) -> tuple[float, float]:
    # Load the CSV data into a pandas DataFrame
    df = pd.read_csv(filename)
    # print(df.columns)
    # Convert the 'timestamp' column to datetime
    df['timestamp'] = pd.to_datetime(df['timestamp'])

    # Sort by timestamp just in case it's not sorted
    df = df.sort_values('timestamp')

    # Calculate the time difference between each timestamp
    df['time_diff'] = df['timestamp'].diff().dt.total_seconds()

    # Ensure the memory used column is numeric
    df[metric_name] = df[metric_name].map(lambda x: x.replace('%', ''))
    df[metric_name] = df[metric_name].map(lambda x: x.replace('MiB', ''))
    df[metric_name] = df[metric_name].map(lambda x: x.replace(' ', ''))
    if 'memory.used' in metric_name:
        df[' memory.total [MiB]'] = df[' memory.total [MiB]'].map(lambda x: x.replace('%', ''))
        df[' memory.total [MiB]'] = df[' memory.total [MiB]'].map(lambda x: x.replace('MiB', ''))
        df[' memory.total [MiB]'] = df[' memory.total [MiB]'].map(lambda x: x.replace(' ', ''))
        df[metric_name] = df[metric_name].astype(float) / df[' memory.total [MiB]'].astype(float) * 100
    df[metric_name] = pd.to_numeric(df[metric_name], errors='coerce')

    # Multiply memory used by time difference and sum it up
    total_memory = (df[metric_name] * df['time_diff']).sum()

    # Get total time duration
    total_time = df['time_diff'].sum()

    # Compute the time-weighted average
    average_memory_used = total_memory / total_time

    # Compute the time-weighted variance
    df['squared_diff'] = df['time_diff'] * (df[metric_name] - average_memory_used)**2
    variance = df['squared_diff'].sum() / total_time

    # Compute the time-weighted standard deviation of the mean
    std_dev = np.sqrt(variance) / len(df[metric_name])
    print(f"File inspected is {filename}")
    print(f'The time-weighted average of the {metric_name} is: {average_memory_used}')
    print(f'The time-weighted standard deviation of the mean of the {metric_name} is: {std_dev}')
    return average_memory_used, std_dev

In [35]:
def get_framework(filename: str) -> str:
    """Get the framework from the filename."""
    frmw = (filename.split('_')[0]).capitalize()
    frmw = "FedScale" if frmw == "Fedscale" else frmw
    frmw = "Parrot" if frmw == "Pollen-llb" else frmw
    frmw = "Pollen" if frmw == "Pollen-lb" else frmw
    return frmw

def get_dataset(filename: str) -> str:
    """Get the dataset from the filename."""
    if 'openimage' in filename:
        return 'IC'
    elif 'shake' in filename:
        return 'TG'
    elif 'google' in filename:
        return 'SR'
    else:
        return 'MLM'

In [44]:
res_dict = {
    "Framework": [],
    "Task": [],
    "GPU Utilization": [],
    "GPU Utilization Std Dev": [],
    "GPU Memory": [],
    "GPU Memory Std Dev": [],
}
for file in (HOME_DIR/"projects"/"pollen_worker"/"gpu_results").glob("*.csv"):
    framework = get_framework(file.name)
    dataset = get_dataset(file.name)
    util, util_stddev = get_time_avg_metrics(file, ' utilization.gpu [%]')
    mem, mem_stddev = get_time_avg_metrics(file, ' memory.used [MiB]')
    # t_mem, t_mem_stddev = get_time_avg_metrics(file, ' memory.total [MiB]')
    res_dict["Framework"].append(framework)
    res_dict["Task"].append(dataset)
    res_dict["GPU Utilization"].append(util)
    res_dict["GPU Utilization Std Dev"].append(util_stddev)
    # res_dict["GPU Memory"].append(mem/t_mem)
    # res_dict["GPU Memory Std Dev"].append(mem_stddev/t_mem)
    res_dict["GPU Memory"].append(mem)
    res_dict["GPU Memory Std Dev"].append(mem_stddev)
df = pd.DataFrame(res_dict)

File inspected is /nfs-share/ls985/projects/pollen_worker/gpu_results/pollen-lb_monitor_reddit_2024-01-22_182624_73bda201-d5a3-4197-ba31-d807c986afcf.csv
The time-weighted average of the  utilization.gpu [%] is: 76.73447746616435
The time-weighted standard deviation of the mean of the  utilization.gpu [%] is: 0.003955210935612087
File inspected is /nfs-share/ls985/projects/pollen_worker/gpu_results/pollen-lb_monitor_reddit_2024-01-22_182624_73bda201-d5a3-4197-ba31-d807c986afcf.csv
The time-weighted average of the  memory.used [MiB] is: 67.62384524060231
The time-weighted standard deviation of the mean of the  memory.used [MiB] is: 0.0016564114029286167
File inspected is /nfs-share/ls985/projects/pollen_worker/gpu_results/flute_monitor_shakespeare_2024-01-22 18:50:29.303517_mauao.csv
The time-weighted average of the  utilization.gpu [%] is: 21.987406591443627
The time-weighted standard deviation of the mean of the  utilization.gpu [%] is: 0.001099246409908866
File inspected is /nfs-shar

In [45]:
def round_to_first_sig_fig(x):
    if x != 0:
        return round(x, -int(np.floor(np.log10(abs(x)))))
    else:
        return 0
def bold(s):
    return f'\\textbf{{{s}}}'

def bold_max(dataframe):
    max_val = dataframe.max().max()
    return dataframe.applymap(lambda x: f'\\textbf{{{x}}}' if x == max_val else x)

def bold_max_row(dataframe):
    max_val = dataframe.max(axis=1)
    return dataframe.apply(lambda x: ['\\textbf{' + str(v) + '}' if v == max_val[i] else str(v) for i, v in enumerate(x)], axis=1)


In [46]:
round_to_first_sig_fig(0.01)

0.01

In [47]:
metrics = ["GPU Utilization", "GPU Memory"]

for metric in metrics:
    mean_df = df.pivot(index='Task', columns='Framework', values=f'{metric}')
    std_df = df.pivot(index='Task', columns='Framework', values=f'{metric} Std Dev')
    
    # mean_df = mean_df.applymap(round_to_first_sig_fig)
    std_df = std_df.applymap(round_to_first_sig_fig)

    combined_df = mean_df.astype(str) + " ± " + std_df.astype(str)
    combined_df = bold_max(combined_df)
    # combined_df = bold_max_row(combined_df)

    latex_table = combined_df.to_latex(escape=False)

    print(f"LaTeX table for {metric}:\n{latex_table}\n")

LaTeX table for GPU Utilization:
\begin{tabular}{llllll}
\toprule
Framework & FedScale & Flower & Flute & Parrot & Pollen \\
Task &  &  &  &  &  \\
\midrule
IC & 19.023498634069547 ± 0.002 & 66.98654577361802 ± 0.005 & 13.750923602822366 ± 0.0003 & 16.030089796247076 ± 0.0005 & \textbf{95.28141042961362 ± 0.003} \\
MLM & 4.549303269093179 ± 6e-05 & 83.31400915355881 ± 0.004 & 22.282493493462102 ± 0.0006 & 14.682718025712669 ± 0.0004 & 76.73447746616435 ± 0.004 \\
SR & 4.310975801877874 ± 0.0002 & 19.66582685215613 ± 0.001 & 4.842150256852641 ± 9e-05 & 2.4849574399899947 ± 3e-05 & 20.673732682307172 ± 0.004 \\
TG & 35.2813056379822 ± 0.02 & 86.4046037019459 ± 0.02 & 21.987406591443627 ± 0.001 & 32.45661878881987 ± 0.007 & 85.81138613861386 ± 0.03 \\
\bottomrule
\end{tabular}


LaTeX table for GPU Memory:
\begin{tabular}{llllll}
\toprule
Framework & FedScale & Flower & Flute & Parrot & Pollen \\
Task &  &  &  &  &  \\
\midrule
IC & 89.9366760452907 ± 0.0003 & 89.65757100348598 ± 0.002 & 

/tmp/ipykernel_1849324/2602118995.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  std_df = std_df.applymap(round_to_first_sig_fig)
/tmp/ipykernel_1849324/103472173.py:11: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return dataframe.applymap(lambda x: f'\\textbf{{{x}}}' if x == max_val else x)
/tmp/ipykernel_1849324/2602118995.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  std_df = std_df.applymap(round_to_first_sig_fig)
/tmp/ipykernel_1849324/103472173.py:11: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return dataframe.applymap(lambda x: f'\\textbf{{{x}}}' if x == max_val else x)
